# Diabetes — Model Training
Trains several models on the preprocessed data, compares them,
and saves the best one as `models/diabetes_model.pkl`.

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed — skipping it. Run: pip install xgboost")


## 1. Load the preprocessed train/test data
These files were saved at the end of `02_preprocessing.ipynb`.

In [2]:
X_train = pd.read_csv("../../data/processed/diabetes_X_train.csv")
X_test = pd.read_csv("../../data/processed/diabetes_X_test.csv")
y_train = pd.read_csv("../../data/processed/diabetes_y_train.csv").squeeze()
y_test = pd.read_csv("../../data/processed/diabetes_y_test.csv").squeeze()

print("X_train:", X_train.shape, "| X_test:", X_test.shape)


X_train: (614, 8) | X_test: (154, 8)


## 2. Define the models to compare
Starting with simple, interpretable models (important for a
*clinical* system) before trying more complex ones.

In [3]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

if HAS_XGB:
    models["XGBoost"] = XGBClassifier(
        use_label_encoder=False, eval_metric="logloss", random_state=42
    )


## 3. Train and evaluate each model
We track accuracy, precision, recall, F1, and ROC-AUC —
not just accuracy, since missing a diabetic patient (false
negative) matters more than a false alarm.

In [4]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
results_df


c:\Users\SADAB EHTESHAM\Desktop\m project\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:20:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
2,Random Forest,0.740260,0.652174,0.555556,0.600000,0.816111
3,XGBoost,0.772727,0.702128,0.611111,0.653465,0.815370
0,Logistic Regression,0.707792,0.600000,0.500000,0.545455,0.812963
1,Decision Tree,0.681818,0.553191,0.481481,0.514851,0.635741


## 4. Pick the best model
By default we pick the highest ROC-AUC — a good overall
measure for imbalanced medical classification tasks.
You can change `best_model_name` manually if you prefer a
different model (e.g. for interpretability).

In [5]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print(f"Best model: {best_model_name}")


Best model: Random Forest


## 5. Confusion matrix for the best model
Shows exactly how many patients were correctly/incorrectly
classified — the diagonal is what you want to be large.

In [6]:
y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

print("Confusion Matrix (rows = actual, columns = predicted):")
print(cm)
print()
print(f"True Negatives:  {cm[0][0]}   False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}   True Positives:  {cm[1][1]}")


Confusion Matrix (rows = actual, columns = predicted):
[[84 16]
 [24 30]]

True Negatives:  84   False Positives: 16
False Negatives: 24   True Positives:  30


## 6. Save the best model
This `.pkl` file is what the Flask app will load later
at prediction time — no retraining needed on the website.

In [7]:
joblib.dump(best_model, "../../models/diabetes_model.pkl")
print("Saved: models/diabetes_model.pkl")


Saved: models/diabetes_model.pkl


## Next step
Open **04_evaluation.ipynb** for a deeper look at this model's
performance (ROC curve, feature importance), or move on to
repeating this same notebook pattern for the next disease.